# Module 03 — Unsupervised & Statistical Learning
## 03-03: Principal Component Analysis

**Objective:** Implement PCA from scratch via eigendecomposition of the covariance matrix, apply it for visualization, compression, and denoising, and prove its equivalence to truncated SVD.

**Prerequisites:** 1-06 (Linear Algebra Foundations)

## Part 0 — Setup & Prerequisites

PCA is the most widely used dimensionality reduction algorithm in ML. Given high-dimensional data, PCA finds a lower-dimensional subspace that captures the maximum variance by computing the eigenvectors of the data covariance matrix.

In this notebook we:
1. Implement PCA step-by-step from the covariance eigendecomposition.
2. Wrap everything into a reusable `PCA` class with `fit` / `transform` / `inverse_transform`.
3. Apply PCA to Digits (8×8) and MNIST (28×28) for 2-D / 3-D visualization and denoising.
4. Prove the mathematical equivalence between eigendecomposition PCA and truncated SVD.
5. Demonstrate PCA's core limitation: it only captures *linear* structure.

**Prerequisites:** 1-06 (Linear Algebra Foundations)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401 — registers 3-D projection

from sklearn.datasets import load_digits, make_moons
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import torch
import torchvision
import torchvision.transforms as transforms

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
import random

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Configuration ─────────────────────────────────────────────────────────────
N_COMPONENTS_DIGITS = 64     # max for 8x8 digits
N_COMPONENTS_MNIST  = 100    # truncated sweep for MNIST
NOISE_STD           = 0.5    # Gaussian noise std for denoising demo
DATA_DIR            = "../data"

print(f"Seed: {SEED}  |  Noise std: {NOISE_STD}")

### Data Loading & EDA

We use two datasets:
- **Digits** (`sklearn`): 1 797 samples, 64 features (8×8 grayscale images of digits 0–9).
- **MNIST** (torchvision): We load a 10 000-sample subset for speed.

In [ ]:
# ── Load Digits ───────────────────────────────────────────────────────────────
digits   = load_digits()
X_digits = digits.data.astype(np.float64)   # (1797, 64)
y_digits = digits.target

print("Digits dataset:")
print(f"  Shape:         {X_digits.shape}")
print(f"  Classes:       {np.unique(y_digits)}")
print(f"  Feature range: [{X_digits.min():.1f}, {X_digits.max():.1f}]")

# ── EDA: sample images ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit_class in range(10):
    indices = np.where(y_digits == digit_class)[0]
    for row, idx in enumerate(indices[:2]):
        axes[row, digit_class].imshow(
            X_digits[idx].reshape(8, 8), cmap="gray_r"
        )
        if row == 0:
            axes[row, digit_class].set_title(str(digit_class), fontsize=9)
        axes[row, digit_class].axis("off")
plt.suptitle("Sample Digits — Two Examples per Class", fontsize=10)
plt.tight_layout()
plt.show()

# ── Train / test split (80/20) ────────────────────────────────────────────────
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=SEED, stratify=y_digits
)
print(f"\nDigits split — train: {X_train_d.shape}  test: {X_test_d.shape}")

In [ ]:
# ── Load MNIST subset (10 000 samples) ───────────────────────────────────────
mnist_full = torchvision.datasets.MNIST(
    root=DATA_DIR, train=True, download=True,
    transform=transforms.ToTensor(),
)
rng_data = np.random.RandomState(SEED)
mnist_idx = rng_data.choice(len(mnist_full), size=10_000, replace=False)
mnist_idx.sort()

X_mnist = np.array(
    [mnist_full[int(i)][0].numpy().flatten() for i in mnist_idx],
    dtype=np.float64,
)
y_mnist = np.array([mnist_full[int(i)][1] for i in mnist_idx])

print(f"MNIST subset — shape: {X_mnist.shape}")
print(f"Pixel range: [{X_mnist.min():.3f}, {X_mnist.max():.3f}]")
print(f"Class counts: {np.bincount(y_mnist).tolist()}")

---
## Part 1 — PCA from Scratch

### 1.1 Mathematical Foundation

Given a data matrix $\mathbf{X} \in \mathbb{R}^{n \times d}$, PCA finds an orthonormal basis $\mathbf{V}_k \in \mathbb{R}^{d \times k}$ (the top-$k$ principal components) that maximises the variance of the projected data $\mathbf{Z} = \tilde{\mathbf{X}} \mathbf{V}_k$.

**Step 1 — Center:**
$$\bar{\mathbf{x}} = \frac{1}{n}\sum_{i=1}^n \mathbf{x}_i, \qquad \tilde{\mathbf{X}} = \mathbf{X} - \mathbf{1}\bar{\mathbf{x}}^\top$$

**Step 2 — Covariance matrix:**
$$\mathbf{C} = \frac{1}{n-1}\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} \in \mathbb{R}^{d \times d}$$

**Step 3 — Eigendecomposition:**
$$\mathbf{C}\mathbf{v}_j = \lambda_j \mathbf{v}_j, \qquad \lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_d \geq 0$$

Eigenvalue $\lambda_j$ equals the variance of the data projected onto $\mathbf{v}_j$.

**Step 4 — Project and reconstruct:**
$$\mathbf{Z} = \tilde{\mathbf{X}}\mathbf{V}_k, \qquad \hat{\mathbf{X}} = \mathbf{Z}\mathbf{V}_k^\top + \bar{\mathbf{x}}$$

The reconstruction error $\|\mathbf{X} - \hat{\mathbf{X}}\|_F^2$ decreases monotonically with $k$.

In [ ]:
def center_data(X: np.ndarray) -> tuple:
    """Subtract the feature-wise mean to zero-center the data matrix.

    Args:
        X: Data matrix of shape (n_samples, n_features).

    Returns:
        Tuple of (X_centered, mean_vector). X_centered shape: (n_samples, n_features).
        mean_vector shape: (n_features,).
    """
    mean_vector = X.mean(axis=0)
    X_centered  = X - mean_vector
    return X_centered, mean_vector


# ── Verify on toy example ─────────────────────────────────────────────────────
X_toy = np.array([[1.0, 4.0], [3.0, 2.0], [5.0, 6.0]])
X_c_toy, mu_toy = center_data(X_toy)
print("Centering check:")
print(f"  Original:        {X_toy.tolist()}")
print(f"  Mean:            {mu_toy.tolist()}")
print(f"  Centered:        {X_c_toy.tolist()}")
print(f"  Column means ~0: {np.allclose(X_c_toy.mean(axis=0), 0)}")

In [ ]:
def compute_covariance_matrix(X_centered: np.ndarray) -> np.ndarray:
    """Compute the unbiased sample covariance matrix of a centered data matrix.

    Uses the formula C = X^T X / (n - 1).

    Args:
        X_centered: Zero-centered matrix of shape (n_samples, n_features).

    Returns:
        Symmetric covariance matrix of shape (n_features, n_features).
    """
    n_samples = X_centered.shape[0]
    return (X_centered.T @ X_centered) / (n_samples - 1)


# ── Apply to Digits training set ──────────────────────────────────────────────
X_c_digits, mu_digits = center_data(X_train_d)
cov_digits = compute_covariance_matrix(X_c_digits)

print(f"Covariance matrix shape: {cov_digits.shape}")
print(f"Symmetric (max |C - C^T|): {np.abs(cov_digits - cov_digits.T).max():.2e}")
print(f"Diagonal (variances) — min: {cov_digits.diagonal().min():.4f}  "
      f"max: {cov_digits.diagonal().max():.4f}")
psd_check = np.all(np.linalg.eigvalsh(cov_digits) >= -1e-10)
print(f"Positive semi-definite (all eigenvalues >= 0): {psd_check}")

In [ ]:
def eigendecompose_covariance(cov_matrix: np.ndarray) -> tuple:
    """Eigendecompose a symmetric covariance matrix, sorted by descending eigenvalue.

    Uses numpy.linalg.eigh (optimised for symmetric matrices).
    Clips small negative eigenvalues arising from floating-point errors.

    Args:
        cov_matrix: Symmetric PSD matrix of shape (d, d).

    Returns:
        Tuple (eigenvalues, eigenvectors):
          eigenvalues:  shape (d,), sorted descending.
          eigenvectors: shape (d, d), columns are eigenvectors.
    """
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    # eigh returns ascending order — reverse
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues  = np.clip(eigenvalues[idx], 0.0, None)
    eigenvectors = eigenvectors[:, idx]
    return eigenvalues, eigenvectors


eigenvalues_d, eigenvectors_d = eigendecompose_covariance(cov_digits)

print(f"Eigenvalues shape:   {eigenvalues_d.shape}")
print(f"Eigenvectors shape:  {eigenvectors_d.shape}")
print(f"Top-10 eigenvalues:  {eigenvalues_d[:10].round(2)}")
print(f"All non-negative:    {np.all(eigenvalues_d >= 0)}")
ortho_err = np.abs(eigenvectors_d.T @ eigenvectors_d - np.eye(64)).max()
print(f"Orthonormality error (max |V^T V - I|): {ortho_err:.2e}")

In [ ]:
def compute_explained_variance_ratio(eigenvalues: np.ndarray) -> tuple:
    """Convert eigenvalues to per-component and cumulative explained variance ratios.

    Each eigenvalue lambda_j equals the variance of the data projected onto
    the j-th eigenvector. The EVR is lambda_j / sum(lambda).

    Args:
        eigenvalues: Sorted descending eigenvalue array of shape (d,).

    Returns:
        Tuple (per_evr, cumulative_evr), both shape (d,).
    """
    total_variance = eigenvalues.sum()
    per_evr        = eigenvalues / total_variance
    cum_evr        = np.cumsum(per_evr)
    return per_evr, cum_evr


per_evr_d, cum_evr_d = compute_explained_variance_ratio(eigenvalues_d)

print("Digits — Explained Variance (training set):")
for k_show in [1, 2, 5, 10, 20, 40]:
    print(f"  PC {k_show:<3}: individual {per_evr_d[k_show-1]:.4f} "
          f"| cumulative {cum_evr_d[k_show-1]:.4f}")

k_90 = int(np.searchsorted(cum_evr_d, 0.90)) + 1
k_95 = int(np.searchsorted(cum_evr_d, 0.95)) + 1
print(f"\n  Components to explain 90%: k = {k_90}")
print(f"  Components to explain 95%: k = {k_95}")

In [ ]:
def plot_scree(
    per_evr: np.ndarray,
    cum_evr: np.ndarray,
    dataset_name: str,
    n_show: int = 30,
    thresholds: list | None = None,
) -> None:
    """Plot individual and cumulative explained variance ratios (scree plot).

    Args:
        per_evr: Per-component explained variance ratio, shape (d,).
        cum_evr: Cumulative explained variance ratio, shape (d,).
        dataset_name: Label for plot titles.
        n_show: Number of components to show on the x-axis.
        thresholds: List of cumulative EVR thresholds to mark (default [0.90, 0.95]).
    """
    if thresholds is None:
        thresholds = [0.90, 0.95]
    threshold_colors = ["red", "navy"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    x = np.arange(1, n_show + 1)

    axes[0].bar(x, per_evr[:n_show] * 100,
                color="#4472C4", edgecolor="black", linewidth=0.4)
    axes[0].set_xlabel("Principal Component")
    axes[0].set_ylabel("Explained Variance (%)")
    axes[0].set_title(f"Individual EVR — {dataset_name}")
    axes[0].set_xticks(x[::max(1, n_show // 10)])

    axes[1].plot(x, cum_evr[:n_show] * 100, "o-", color="#ED7D31", markersize=4)
    for thresh, color in zip(thresholds, threshold_colors):
        k_t = int(np.searchsorted(cum_evr, thresh)) + 1
        axes[1].axhline(y=thresh * 100, color=color, linestyle="--",
                        label=f"{thresh:.0%} at k={k_t}")
        axes[1].axvline(x=k_t, color=color, linestyle=":", alpha=0.5)
    axes[1].set_xlabel("Number of Components (k)")
    axes[1].set_ylabel("Cumulative Explained Variance (%)")
    axes[1].set_title(f"Cumulative EVR — {dataset_name}")
    axes[1].legend(fontsize=8)
    axes[1].set_ylim(0, 105)
    axes[1].set_xticks(x[::max(1, n_show // 10)])

    plt.tight_layout()
    plt.show()


plot_scree(per_evr_d, cum_evr_d, "Digits (8×8 pixels)")

In [ ]:
def project_to_subspace(
    X_centered: np.ndarray,
    components: np.ndarray,
) -> np.ndarray:
    """Project zero-centered data onto the PCA subspace defined by components.

    Args:
        X_centered: Centered data of shape (n_samples, n_features).
        components: Row-stacked eigenvectors of shape (k, n_features).

    Returns:
        Projected scores of shape (n_samples, k).
    """
    return X_centered @ components.T   # (n, d) @ (d, k) = (n, k)


def reconstruct_from_subspace(
    Z: np.ndarray,
    components: np.ndarray,
    mean_vector: np.ndarray,
) -> np.ndarray:
    """Reconstruct original-space data from PCA projected scores.

    Args:
        Z: Projected scores of shape (n_samples, k).
        components: Row-stacked eigenvectors of shape (k, n_features).
        mean_vector: Per-feature mean, shape (n_features,).

    Returns:
        Reconstructed data of shape (n_samples, n_features).
    """
    return Z @ components + mean_vector   # (n, k) @ (k, d) + (d,)


# ── Quick test on Digits ──────────────────────────────────────────────────────
k_test = 20
components_k = eigenvectors_d[:, :k_test].T   # (k, 64)
Z_test_proj  = project_to_subspace(X_c_digits, components_k)
X_recon_test = reconstruct_from_subspace(Z_test_proj, components_k, mu_digits)

mse_k = np.mean((X_train_d - X_recon_test) ** 2)
print(f"Projection test (k={k_test}):")
print(f"  Z shape:         {Z_test_proj.shape}")
print(f"  Recon shape:     {X_recon_test.shape}")
print(f"  Recon MSE:       {mse_k:.4f}")
print(f"  Cumulative EVR:  {cum_evr_d[k_test - 1]:.4f}")

---
## Part 2 — Putting It All Together: the `PCA` Class

We wrap all steps into a scikit-learn-compatible class with `fit`, `transform`,
`inverse_transform`, and `fit_transform`. An optional `whiten` parameter scales
each PC to unit variance, which is useful as preprocessing for ICA (03-05)
or k-Means.

In [ ]:
class PCA:
    """Principal Component Analysis via covariance matrix eigendecomposition.

    Implements the same interface as sklearn.decomposition.PCA and operates
    on NumPy arrays. Supports optional whitening.

    Attributes:
        n_components: Number of principal components to retain.
        whiten: If True, scale projected scores to unit variance per PC.
        components_: Eigenvectors as rows, shape (n_components, n_features).
        explained_variance_: Top-k eigenvalues, shape (n_components,).
        explained_variance_ratio_: Fraction of total variance per component.
        cumulative_variance_ratio_: Cumulative EVR, shape (n_components,).
        mean_: Per-feature mean of training data, shape (n_features,).
        singular_values_: sqrt((n-1) * eigenvalues), shape (n_components,).
        is_fitted_: Whether fit() has been called.
    """

    def __init__(
        self,
        n_components: int = 2,
        whiten: bool = False,
    ) -> None:
        """Initialise PCA.

        Args:
            n_components: Number of principal components to keep.
            whiten: If True, divide projected scores by their std so each PC
                    has unit variance.
        """
        self.n_components = n_components
        self.whiten       = whiten
        self.is_fitted_   = False

    def fit(self, X: np.ndarray) -> "PCA":
        """Fit PCA to training data: compute mean, covariance, and eigenvectors.

        Args:
            X: Training data of shape (n_samples, n_features).

        Returns:
            Self (for method chaining).
        """
        n_samples          = X.shape[0]
        X_centered, self.mean_ = center_data(X)
        cov                = compute_covariance_matrix(X_centered)
        eigenvalues, eigenvectors = eigendecompose_covariance(cov)

        self.components_               = eigenvectors[:, :self.n_components].T
        self.explained_variance_       = eigenvalues[:self.n_components]
        total_var                      = eigenvalues.sum()
        self.explained_variance_ratio_ = self.explained_variance_ / total_var
        self.cumulative_variance_ratio_ = np.cumsum(self.explained_variance_ratio_)
        self.singular_values_          = np.sqrt(
            (n_samples - 1) * np.clip(self.explained_variance_, 0, None)
        )
        self.n_features_in_            = X.shape[1]
        self.is_fitted_                = True
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Project data onto the fitted PCA subspace.

        Args:
            X: Data of shape (n_samples, n_features).

        Returns:
            Projected scores of shape (n_samples, n_components).
        """
        if not self.is_fitted_:
            raise RuntimeError("Call fit() before transform().")
        X_centered = X - self.mean_
        Z = project_to_subspace(X_centered, self.components_)
        if self.whiten:
            Z = Z / (np.sqrt(self.explained_variance_) + 1e-10)
        return Z

    def inverse_transform(self, Z: np.ndarray) -> np.ndarray:
        """Reconstruct original-space data from projected scores.

        Args:
            Z: Projected scores of shape (n_samples, n_components).

        Returns:
            Reconstructed data of shape (n_samples, n_features).
        """
        if not self.is_fitted_:
            raise RuntimeError("Call fit() before inverse_transform().")
        Z_unwhitened = Z
        if self.whiten:
            Z_unwhitened = Z * (np.sqrt(self.explained_variance_) + 1e-10)
        return reconstruct_from_subspace(Z_unwhitened, self.components_, self.mean_)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        """Fit PCA and immediately project the training data.

        Args:
            X: Training data of shape (n_samples, n_features).

        Returns:
            Projected training scores of shape (n_samples, n_components).
        """
        return self.fit(X).transform(X)

    def __repr__(self) -> str:
        return (f"PCA(n_components={self.n_components}, "
                f"whiten={self.whiten}, fitted={self.is_fitted_})")

In [ ]:
# ── Sanity Check ─────────────────────────────────────────────────────────────
# With n_components = n_features the reconstruction should be exact.
X_toy2 = np.array([[2.5, 2.4], [0.5, 0.7], [2.2, 2.9],
                   [1.9, 2.2], [3.1, 3.0], [2.3, 2.7],
                   [2.0, 1.6], [1.0, 1.1], [1.5, 1.6], [1.1, 0.9]])

pca_full_toy = PCA(n_components=2)
Z_full_toy   = pca_full_toy.fit_transform(X_toy2)
X_rec_toy    = pca_full_toy.inverse_transform(Z_full_toy)

print("Sanity check (2-D toy data, k=d=2):")
print(f"  EVR per component: {pca_full_toy.explained_variance_ratio_.round(4)}")
print(f"  Cumulative EVR:    {pca_full_toy.cumulative_variance_ratio_.round(4)}")
print(f"  Reconstruction MSE (should be ~0): {np.mean((X_toy2 - X_rec_toy)**2):.2e}")

# k < d should give non-zero MSE
pca_k1 = PCA(n_components=1)
Z_k1   = pca_k1.fit_transform(X_toy2)
X_rec_k1 = pca_k1.inverse_transform(Z_k1)
print(f"  Reconstruction MSE (k=1 < d=2):   {np.mean((X_toy2 - X_rec_k1)**2):.4f}")
print(f"\n{pca_full_toy}")

---
## Part 3 — Application on Digits and MNIST

### 3.1 2-D Visualization

Projecting to 2 principal components produces a 2-D map. Points of the same
digit class should cluster together if PCA captures discriminative structure.

In [ ]:
# ── 2-D PCA on Digits ────────────────────────────────────────────────────────
pca_2d   = PCA(n_components=2)
Z_2d_tr  = pca_2d.fit_transform(X_train_d)
Z_2d_te  = pca_2d.transform(X_test_d)

print(f"2-D PCA — PC1 EVR: {pca_2d.explained_variance_ratio_[0]:.4f}  "
      f"PC2 EVR: {pca_2d.explained_variance_ratio_[1]:.4f}  "
      f"Total: {pca_2d.cumulative_variance_ratio_[1]:.4f}")

colors = plt.cm.tab10(np.linspace(0, 1, 10))
fig, ax = plt.subplots(figsize=(9, 7))
for digit_class in range(10):
    mask = y_train_d == digit_class
    ax.scatter(Z_2d_tr[mask, 0], Z_2d_tr[mask, 1],
               s=12, alpha=0.6, color=colors[digit_class], label=str(digit_class))
ax.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("Digits — 2-D PCA Projection (training set)")
ax.legend(title="Digit", loc="upper right", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print("PC1 roughly separates pen-stroke-heavy digits (0, 8) from thin strokes (1, 7).")

In [ ]:
# ── 3-D PCA on Digits ────────────────────────────────────────────────────────
pca_3d     = PCA(n_components=3)
Z_3d_tr    = pca_3d.fit_transform(X_train_d)

print(f"3-D PCA cumulative EVR: {pca_3d.cumulative_variance_ratio_[2]:.4f}")

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection="3d")
for digit_class in range(10):
    mask = y_train_d == digit_class
    ax.scatter(Z_3d_tr[mask, 0], Z_3d_tr[mask, 1], Z_3d_tr[mask, 2],
               s=8, alpha=0.5, color=colors[digit_class], label=str(digit_class))
ax.set_xlabel(f"PC1 ({pca_3d.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca_3d.explained_variance_ratio_[1]:.1%})")
ax.set_zlabel(f"PC3 ({pca_3d.explained_variance_ratio_[2]:.1%})")
ax.set_title("Digits — 3-D PCA Projection")
ax.legend(title="Digit", loc="upper left", fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

### 3.2 Reconstruction Quality vs Number of Components

Increasing $k$ from 2 to 64 recovers progressively more detail.
We visualise one example per digit class at $k = 2, 5, 10, 20, 40, 64$.

In [ ]:
def reconstruct_at_k(
    X_train: np.ndarray,
    X_show: np.ndarray,
    k_values: list,
    image_shape: tuple,
    vmin: float = 0.0,
    vmax: float = 16.0,
) -> None:
    """Display original images next to PCA reconstructions at various k values.

    Args:
        X_train: Training data used to fit each PCA model.
        X_show: Images to reconstruct and display, shape (n_show, n_features).
        k_values: List of k values (number of PCA components) to evaluate.
        image_shape: Tuple (height, width) for reshaping flattened pixels.
        vmin: Colourmap minimum value.
        vmax: Colourmap maximum value.
    """
    n_show = len(X_show)
    n_cols = len(k_values) + 1   # +1 for original

    fig, axes = plt.subplots(n_show, n_cols, figsize=(n_cols * 1.6, n_show * 1.6))

    for row_idx in range(n_show):
        axes[row_idx, 0].imshow(
            X_show[row_idx].reshape(image_shape), cmap="gray_r", vmin=vmin, vmax=vmax
        )
        axes[row_idx, 0].axis("off")
        if row_idx == 0:
            axes[row_idx, 0].set_title("Original", fontsize=8)

        for col_idx, k in enumerate(k_values, start=1):
            pca_k = PCA(n_components=k)
            pca_k.fit(X_train)
            Z_k   = pca_k.transform(X_show[row_idx:row_idx + 1])
            X_rec = pca_k.inverse_transform(Z_k)
            axes[row_idx, col_idx].imshow(
                X_rec[0].reshape(image_shape), cmap="gray_r", vmin=vmin, vmax=vmax
            )
            axes[row_idx, col_idx].axis("off")
            if row_idx == 0:
                mse_k = np.mean((X_show[row_idx] - X_rec[0]) ** 2)
                axes[row_idx, col_idx].set_title(f"k={k}\nMSE={mse_k:.2f}", fontsize=7)

    plt.suptitle("PCA Reconstruction Quality vs k — Digits", fontsize=10)
    plt.tight_layout()
    plt.show()


sample_rows = [np.where(y_train_d == d)[0][0] for d in range(5)]
X_sample = X_train_d[sample_rows]
reconstruct_at_k(X_train_d, X_sample, k_values=[2, 5, 10, 20, 40, 64],
                 image_shape=(8, 8))

### 3.3 PCA for Image Denoising

PCA denoising exploits the fact that signal lives in the high-variance top-$k$
subspace while noise spreads across all directions. Projecting noisy data onto the
top-$k$ components and back discards noise concentrated in the discarded directions.

In [ ]:
def add_gaussian_noise(
    X: np.ndarray,
    noise_std: float,
    seed: int = SEED,
) -> np.ndarray:
    """Add independent Gaussian noise to every element of X.

    Args:
        X: Clean data matrix of shape (n_samples, n_features).
        noise_std: Standard deviation of added Gaussian noise.
        seed: Random seed for reproducibility.

    Returns:
        Noisy data matrix of shape (n_samples, n_features).
    """
    rng = np.random.RandomState(seed)
    return X + rng.randn(*X.shape) * noise_std


def pca_denoise(
    X_noisy: np.ndarray,
    n_components: int,
    pca_clean: "PCA",
) -> np.ndarray:
    """Denoise samples by projecting onto top-k clean PCA components.

    Fits a k-component PCA from the clean model's top-k eigenvectors,
    then reconstructs to remove the noise in discarded dimensions.

    Args:
        X_noisy: Noisy data of shape (n_samples, n_features).
        n_components: Number of PCA components to retain.
        pca_clean: A PCA object fitted on clean training data with enough components.

    Returns:
        Denoised reconstruction of shape (n_samples, n_features).
    """
    pca_k = PCA(n_components=n_components)
    pca_k.components_         = pca_clean.components_[:n_components]
    pca_k.explained_variance_ = pca_clean.explained_variance_[:n_components]
    pca_k.mean_               = pca_clean.mean_
    pca_k.whiten              = False
    pca_k.is_fitted_          = True
    Z = pca_k.transform(X_noisy)
    return pca_k.inverse_transform(Z)


# Fit full PCA on clean training data
pca_clean_full = PCA(n_components=64)
pca_clean_full.fit(X_train_d)

X_test_noisy = add_gaussian_noise(X_test_d, noise_std=NOISE_STD)
mse_noisy    = np.mean((X_test_d - X_test_noisy) ** 2)
print(f"Denoising experiment (noise_std={NOISE_STD}):")
print(f"  Noisy MSE (no denoising): {mse_noisy:.4f}")

for k_dn in [5, 10, 20, 40]:
    X_den = pca_denoise(X_test_noisy, n_components=k_dn, pca_clean=pca_clean_full)
    mse_den = np.mean((X_test_d - X_den) ** 2)
    print(f"  k={k_dn:<3}  Denoised MSE: {mse_den:.4f}")

In [ ]:
# ── Visualise denoising ───────────────────────────────────────────────────────
k_denoise_values = [5, 10, 20, 40]
n_show_rows = 4

fig, axes = plt.subplots(n_show_rows, len(k_denoise_values) + 2,
                          figsize=(12, 5))

for row_idx in range(n_show_rows):
    idx = row_idx
    axes[row_idx, 0].imshow(X_test_d[idx].reshape(8, 8),
                             cmap="gray_r", vmin=0, vmax=16)
    axes[row_idx, 0].axis("off")
    if row_idx == 0:
        axes[row_idx, 0].set_title("Clean", fontsize=8)

    axes[row_idx, 1].imshow(X_test_noisy[idx].reshape(8, 8),
                             cmap="gray_r", vmin=0, vmax=16)
    axes[row_idx, 1].axis("off")
    if row_idx == 0:
        axes[row_idx, 1].set_title(f"Noisy\n(σ={NOISE_STD})", fontsize=8)

    for col_idx, k_dn in enumerate(k_denoise_values, start=2):
        X_den = pca_denoise(X_test_noisy[idx:idx + 1], n_components=k_dn,
                            pca_clean=pca_clean_full)
        axes[row_idx, col_idx].imshow(X_den[0].reshape(8, 8),
                                      cmap="gray_r", vmin=0, vmax=16)
        axes[row_idx, col_idx].axis("off")
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"k={k_dn}", fontsize=8)

plt.suptitle("PCA Denoising — Clean / Noisy / Denoised (Digits)", fontsize=10)
plt.tight_layout()
plt.show()
print("Small k over-smooths; large k retains noise. k≈20 gives the best trade-off.")

### 3.4 Library Comparison — sklearn PCA

We verify numerical agreement between our `PCA` class and `sklearn.decomposition.PCA`.
Principal components are only defined up to a sign flip, so we align signs before comparing.

In [ ]:
def compare_pca_implementations(
    X_train: np.ndarray,
    X_test: np.ndarray,
    n_components: int,
) -> pd.DataFrame:
    """Compare scratch PCA vs sklearn PCA for EVR, components, and reconstruction.

    Args:
        X_train: Training data, shape (n_train, n_features).
        X_test: Test data, shape (n_test, n_features).
        n_components: Number of PCA components to compare.

    Returns:
        DataFrame comparing fit+transform time and reconstruction MSE.
    """
    # ── Scratch ──────────────────────────────────────────────────────────────
    t0           = time.perf_counter()
    pca_sc       = PCA(n_components=n_components)
    pca_sc.fit(X_train)
    Z_sc         = pca_sc.transform(X_test)
    X_rec_sc     = pca_sc.inverse_transform(Z_sc)
    t_scratch    = time.perf_counter() - t0

    # ── Sklearn ───────────────────────────────────────────────────────────────
    t0           = time.perf_counter()
    pca_sk       = SklearnPCA(n_components=n_components, random_state=SEED)
    pca_sk.fit(X_train)
    Z_sk         = pca_sk.transform(X_test)
    X_rec_sk     = pca_sk.inverse_transform(Z_sk)
    t_sklearn    = time.perf_counter() - t0

    # ── Alignment & comparison ────────────────────────────────────────────────
    evr_diff  = np.abs(pca_sc.explained_variance_ratio_ - pca_sk.explained_variance_ratio_).max()
    signs     = np.sign(np.sum(pca_sc.components_ * pca_sk.components_, axis=1, keepdims=True))
    comp_diff = np.abs(pca_sc.components_ * signs - pca_sk.components_).max()
    mse_sc    = np.mean((X_test - X_rec_sc) ** 2)
    mse_sk    = np.mean((X_test - X_rec_sk) ** 2)

    print(f"Comparison (k={n_components}):")
    print(f"  Max |EVR difference|:          {evr_diff:.2e}")
    print(f"  Max |component diff| (aligned): {comp_diff:.2e}")
    print(f"  Scratch recon MSE:  {mse_sc:.4f}")
    print(f"  Sklearn recon MSE:  {mse_sk:.4f}")

    return pd.DataFrame([
        {"Implementation": "Scratch PCA", "Time (s)": round(t_scratch, 4), "MSE": round(mse_sc, 4)},
        {"Implementation": "Sklearn PCA", "Time (s)": round(t_sklearn, 4), "MSE": round(mse_sk, 4)},
    ])


cmp_df = compare_pca_implementations(X_train_d, X_test_d, n_components=20)
print()
print(cmp_df.to_string(index=False))
print("\nBoth agree to machine precision. Sklearn is faster (uses LAPACK routines).")

### 3.5 2-D PCA on MNIST

We repeat the 2-D projection on MNIST (784 features vs 64 for Digits) to confirm
PCA scales to higher-dimensional data and still reveals class structure.

In [ ]:
pca_mnist_2d  = PCA(n_components=2)
Z_mnist_2d    = pca_mnist_2d.fit_transform(X_mnist)

print(f"MNIST 2-D PCA:")
print(f"  PC1 EVR: {pca_mnist_2d.explained_variance_ratio_[0]:.4f}")
print(f"  PC2 EVR: {pca_mnist_2d.explained_variance_ratio_[1]:.4f}")

fig, ax = plt.subplots(figsize=(10, 8))
for digit_class in range(10):
    mask = y_mnist == digit_class
    ax.scatter(Z_mnist_2d[mask, 0], Z_mnist_2d[mask, 1],
               s=5, alpha=0.4, color=colors[digit_class], label=str(digit_class))
ax.set_xlabel(f"PC1 ({pca_mnist_2d.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca_mnist_2d.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("MNIST — 2-D PCA Projection (10 000 samples)")
ax.legend(title="Digit", loc="upper right", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print("2-D PCA only captures ~15% of MNIST variance; class overlap is expected.")
print("Non-linear methods (t-SNE, UMAP — 03-04) achieve much better separation.")

### 3.6 SVD Equivalence

**Theorem:** PCA via eigendecomposition is mathematically equivalent to truncated SVD.

If $\tilde{\mathbf{X}} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$ is the SVD of the centred data, then:
- The principal components (eigenvectors of $\mathbf{C}$) equal the right singular vectors $\mathbf{V}$.
- The eigenvalues satisfy $\lambda_j = \sigma_j^2 / (n-1)$.

SVD is preferred in practice because it avoids forming the $d \times d$ covariance matrix,
which is critical when $d \gg n$ (more features than samples).

In [ ]:
def pca_via_svd(
    X_train: np.ndarray,
    X_test: np.ndarray,
    n_components: int,
) -> tuple:
    """Compute PCA projections using full SVD on the centred data matrix.

    More numerically stable than forming the covariance matrix explicitly.
    Avoids the d×d covariance matrix when n << d.

    Args:
        X_train: Training data of shape (n_train, n_features).
        X_test: Test data of shape (n_test, n_features).
        n_components: Number of components to retain.

    Returns:
        Tuple (Z_train, Z_test, components_k, evr_k) where:
          Z_train: Projected training scores, shape (n_train, k).
          Z_test:  Projected test scores, shape (n_test, k).
          components_k: Top-k right singular vectors as rows, shape (k, d).
          evr_k: Explained variance ratio for top-k components, shape (k,).
    """
    X_c_train, mean = center_data(X_train)
    U, sigma, Vt    = np.linalg.svd(X_c_train, full_matrices=False)
    n_train         = X_train.shape[0]

    Vt_k            = Vt[:n_components]
    sigma_k         = sigma[:n_components]
    eigenvalues_k   = sigma_k ** 2 / (n_train - 1)
    total_var       = (sigma ** 2).sum() / (n_train - 1)
    evr_k           = eigenvalues_k / total_var

    Z_train = (X_c_train @ Vt_k.T)
    Z_test  = ((X_test - mean) @ Vt_k.T)
    return Z_train, Z_test, Vt_k, evr_k


Z_train_svd, Z_test_svd, comps_svd, evr_svd = pca_via_svd(
    X_train_d, X_test_d, n_components=20
)

# Compare to eigendecomposition PCA
pca_eig20 = PCA(n_components=20)
pca_eig20.fit(X_train_d)
Z_test_eig = pca_eig20.transform(X_test_d)

signs    = np.sign(np.sum(comps_svd * pca_eig20.components_, axis=1, keepdims=True))
c_diff   = np.abs(comps_svd * signs - pca_eig20.components_).max()
z_diff   = np.abs(Z_test_svd * signs.T - Z_test_eig).max()
evr_diff = np.abs(evr_svd - pca_eig20.explained_variance_ratio_).max()

print("SVD vs Eigendecomposition PCA (k=20, Digits):")
print(f"  Max component difference (sign-aligned): {c_diff:.2e}")
print(f"  Max projection difference (sign-aligned): {z_diff:.2e}")
print(f"  Max EVR difference:                       {evr_diff:.2e}")
print("\nBoth are numerically equivalent to machine precision. ✓")

---
## Part 4 — Evaluation & Analysis

### 4.1 Reconstruction Error vs k

We sweep $k$ from 1 to 64 and measure MSE on the held-out test set.

In [ ]:
def reconstruction_error_sweep(
    X_train: np.ndarray,
    X_test: np.ndarray,
    k_values: list,
) -> pd.DataFrame:
    """Compute train and test reconstruction MSE across a range of k values.

    Fits one max-k PCA then evaluates sub-k projections without refitting,
    since eigenvectors are nested (adding k+1 never changes prior eigenvectors).

    Args:
        X_train: Training data, shape (n_train, n_features).
        X_test: Test data, shape (n_test, n_features).
        k_values: List of component counts to evaluate.

    Returns:
        DataFrame with columns [k, Train MSE, Test MSE, Cumulative EVR (%)].
    """
    k_max   = max(k_values)
    pca_max = PCA(n_components=k_max)
    pca_max.fit(X_train)
    rows = []

    for k in k_values:
        pca_k = PCA(n_components=k)
        pca_k.components_         = pca_max.components_[:k]
        pca_k.explained_variance_ = pca_max.explained_variance_[:k]
        pca_k.mean_               = pca_max.mean_
        pca_k.whiten              = False
        pca_k.is_fitted_          = True

        Z_tr    = pca_k.transform(X_train)
        Z_te    = pca_k.transform(X_test)
        X_r_tr  = pca_k.inverse_transform(Z_tr)
        X_r_te  = pca_k.inverse_transform(Z_te)

        rows.append({
            "k":                int(k),
            "Train MSE":        float(np.mean((X_train - X_r_tr) ** 2)),
            "Test MSE":         float(np.mean((X_test  - X_r_te) ** 2)),
            "Cumulative EVR %": float(pca_max.cumulative_variance_ratio_[k - 1] * 100),
        })
    return pd.DataFrame(rows)


k_sweep  = list(range(1, 10)) + list(range(10, 65, 5))
err_df   = reconstruction_error_sweep(X_train_d, X_test_d, k_sweep)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(err_df["k"], err_df["Train MSE"], "o-", color="#4472C4",
             markersize=4, label="Train MSE")
axes[0].plot(err_df["k"], err_df["Test MSE"],  "s-", color="#ED7D31",
             markersize=4, label="Test MSE")
axes[0].set_xlabel("Number of Components (k)")
axes[0].set_ylabel("Reconstruction MSE")
axes[0].set_title("Reconstruction Error vs k — Digits")
axes[0].legend()

axes[1].plot(err_df["k"], err_df["Cumulative EVR %"], "o-", color="#70AD47", markersize=4)
axes[1].axhline(90, color="red",  linestyle="--", label="90%")
axes[1].axhline(95, color="navy", linestyle="--", label="95%")
axes[1].set_xlabel("Number of Components (k)")
axes[1].set_ylabel("Cumulative EVR (%)")
axes[1].set_title("Cumulative EVR vs k — Digits")
axes[1].legend()

plt.tight_layout()
plt.show()

key_ks = err_df[err_df["k"].isin([1, 5, 10, 20, 30, 40, 50, 60, 64])].copy()
print(key_ks.round(3).to_string(index=False))

### 4.2 kNN Classification Accuracy vs Dimensionality

PCA is often used as preprocessing before a classifier. We measure how kNN accuracy
changes as we vary $k$, which reveals the trade-off between dimensionality reduction
and retaining discriminative information.

In [ ]:
def knn_accuracy_vs_k(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    k_pca_values: list,
    k_nn: int = 5,
) -> pd.DataFrame:
    """Measure kNN classification accuracy as a function of PCA dimensionality.

    Args:
        X_train: Training features, shape (n_train, n_features).
        y_train: Training labels, shape (n_train,).
        X_test: Test features, shape (n_test, n_features).
        y_test: Test labels, shape (n_test,).
        k_pca_values: List of PCA component counts to evaluate.
        k_nn: Number of neighbours for kNN (default 5).

    Returns:
        DataFrame with columns [k_pca, Accuracy (%), EVR (%)].
    """
    pca_max = PCA(n_components=max(k_pca_values))
    pca_max.fit(X_train)
    rows    = []

    for k in k_pca_values:
        pca_k = PCA(n_components=k)
        pca_k.components_         = pca_max.components_[:k]
        pca_k.explained_variance_ = pca_max.explained_variance_[:k]
        pca_k.mean_               = pca_max.mean_
        pca_k.whiten              = False
        pca_k.is_fitted_          = True

        Z_tr_k = pca_k.transform(X_train)
        Z_te_k = pca_k.transform(X_test)

        knn = KNeighborsClassifier(n_neighbors=k_nn)
        knn.fit(Z_tr_k, y_train)
        acc = knn.score(Z_te_k, y_test)

        rows.append({
            "k_pca":       int(k),
            "Accuracy (%)": round(acc * 100, 2),
            "EVR (%)":      round(pca_max.cumulative_variance_ratio_[k - 1] * 100, 2),
        })
    return pd.DataFrame(rows)


k_pca_sweep = [1, 2, 3, 5, 8, 10, 15, 20, 30, 40, 50, 64]
acc_df      = knn_accuracy_vs_k(X_train_d, y_train_d, X_test_d, y_test_d, k_pca_sweep)

# Baseline: kNN on raw features
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train_d, y_train_d)
acc_raw = knn_raw.score(X_test_d, y_test_d)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(acc_df["k_pca"], acc_df["Accuracy (%)"], "o-", color="#4472C4", markersize=5)
ax.axhline(acc_raw * 100, color="red", linestyle="--",
           label=f"Raw features (k=64): {acc_raw:.2%}")
ax.set_xlabel("Number of PCA Components (k)")
ax.set_ylabel("kNN Test Accuracy (%)")
ax.set_title("kNN Accuracy vs PCA Dimensionality — Digits")
ax.legend()
plt.tight_layout()
plt.show()

print(acc_df.to_string(index=False))
print(f"\nRaw features (no PCA): {acc_raw:.2%}")
print("PCA with k≈20 matches or exceeds raw-feature accuracy — noise removal helps.")

### 4.3 Visualising the Principal Components (Eigendigits)

Each principal component is a $d$-dimensional vector that can be reshaped into an
image. These "eigendigits" show which patterns capture the most variation in the dataset.

In [ ]:
# ── Plot top-16 principal components as 8×8 images ──────────────────────────
pca_16 = PCA(n_components=16)
pca_16.fit(X_train_d)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
axes = axes.flatten()
for idx in range(16):
    component_img = pca_16.components_[idx].reshape(8, 8)
    axes[idx].imshow(component_img, cmap="coolwarm")
    axes[idx].set_title(
        f"PC {idx+1}\n({pca_16.explained_variance_ratio_[idx]:.1%})", fontsize=7
    )
    axes[idx].axis("off")

plt.suptitle("Top-16 Principal Components — Digits (eigendigits)", fontsize=10)
plt.tight_layout()
plt.show()

print("PC1 (most variance) captures overall brightness / stroke density.")
print("Later PCs capture finer patterns: curves, diagonal strokes, loops.")

# ── Also visualise the mean digit image ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(2.5, 2.5))
ax.imshow(pca_16.mean_.reshape(8, 8), cmap="gray_r")
ax.set_title("Mean digit image", fontsize=10)
ax.axis("off")
plt.tight_layout()
plt.show()

### 4.4 Standardisation — Raw vs Scaled Features

PCA is sensitive to feature scale. When features have very different variances, the
high-variance features dominate the first PCs. `StandardScaler` normalises each feature
to zero mean and unit variance before PCA.

In [ ]:
def compare_raw_vs_scaled_pca(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_components: int = 2,
) -> None:
    """Compare 2-D PCA projections on raw vs StandardScaler-normalised data.

    Args:
        X_train: Training data, shape (n_train, n_features).
        y_train: Training labels.
        X_test: Test data, shape (n_test, n_features).
        y_test: Test labels.
        n_components: Number of PCA components.
    """
    # Raw PCA
    pca_raw = PCA(n_components=n_components)
    Z_raw   = pca_raw.fit_transform(X_train)

    # Scaled PCA
    scaler     = StandardScaler()
    X_tr_sc    = scaler.fit_transform(X_train)
    X_te_sc    = scaler.transform(X_test)
    pca_scaled = PCA(n_components=n_components)
    Z_scaled   = pca_scaled.fit_transform(X_tr_sc)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for cls in range(10):
        m_raw  = y_train == cls
        m_scl  = y_train == cls
        axes[0].scatter(Z_raw[m_raw, 0], Z_raw[m_raw, 1],
                        s=10, alpha=0.5, color=colors[cls], label=str(cls))
        axes[1].scatter(Z_scaled[m_scl, 0], Z_scaled[m_scl, 1],
                        s=10, alpha=0.5, color=colors[cls], label=str(cls))

    for ax, title, pca_obj in zip(
        axes,
        ["Raw features (PCA)", "StandardScaler + PCA"],
        [pca_raw, pca_scaled],
    ):
        ax.set_xlabel(f"PC1 ({pca_obj.explained_variance_ratio_[0]:.1%})")
        ax.set_ylabel(f"PC2 ({pca_obj.explained_variance_ratio_[1]:.1%})")
        ax.set_title(title)
        ax.legend(title="Digit", fontsize=7, ncol=2)

    plt.tight_layout()
    plt.show()

    evr_raw    = pca_raw.cumulative_variance_ratio_[n_components - 1]
    evr_scaled = pca_scaled.cumulative_variance_ratio_[n_components - 1]
    print(f"Cumulative EVR (raw, k={n_components}):    {evr_raw:.4f}")
    print(f"Cumulative EVR (scaled, k={n_components}): {evr_scaled:.4f}")
    print("Digits features have similar scales (0-16), so StandardScaler changes little here.")
    print("For mixed-scale data (e.g., income + age + temperature), scaling is critical.")


compare_raw_vs_scaled_pca(X_train_d, y_train_d, X_test_d, y_test_d, n_components=2)

### 4.5 PCA Limitation — Linearity

PCA only captures *linear* relationships. On non-linearly structured data (e.g., two
interleaved crescents), PCA cannot separate classes even in the optimal projection.
Non-linear manifold methods (t-SNE, UMAP — see 03-04) handle these cases.

In [ ]:
X_moons, y_moons = make_moons(n_samples=600, noise=0.10, random_state=SEED)

pca_moons_1d = PCA(n_components=1)
Z_moons_1d   = pca_moons_1d.fit_transform(X_moons)

pca_moons_2d = PCA(n_components=2)   # identity for 2-D data
Z_moons_2d   = pca_moons_2d.fit_transform(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cls in range(2):
    m = y_moons == cls
    axes[0].scatter(X_moons[m, 0], X_moons[m, 1],
                    s=15, alpha=0.7, label=f"Class {cls}")
    axes[1].scatter(Z_moons_1d[m, 0],
                    np.full(m.sum(), cls * 0.1 + np.random.randn(m.sum()) * 0.02),
                    s=15, alpha=0.5, label=f"Class {cls}")
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")
axes[0].set_title("make_moons — Original 2-D Data")
axes[0].legend()
axes[1].set_xlabel("PC1 Projection")
axes[1].set_title("PCA 1-D Projection — Classes Completely Overlap")
axes[1].legend()
plt.suptitle("PCA Fails on Non-Linear Manifold Structure (make_moons)", fontsize=10)
plt.tight_layout()
plt.show()

# kNN accuracy comparison
knn_orig = KNeighborsClassifier(n_neighbors=5)
knn_orig.fit(X_moons, y_moons)
acc_orig = knn_orig.score(X_moons, y_moons)

knn_pca = KNeighborsClassifier(n_neighbors=5)
knn_pca.fit(Z_moons_1d, y_moons)
acc_pca = knn_pca.score(Z_moons_1d, y_moons)

print(f"kNN accuracy on original 2-D data:       {acc_orig:.4f}")
print(f"kNN accuracy on 1-D PCA projection:      {acc_pca:.4f}")
print("\nConclusion: PCA destroys discriminative structure in non-linear data.")
print("Non-linear methods (t-SNE, UMAP — 03-04) handle this correctly.")

### 4.6 Whitened PCA

Whitening scales each PC to unit variance: $z_j^{\text{white}} = z_j / \sqrt{\lambda_j}$.
This removes the variance differences between components, producing a spherical distribution.
It is required as preprocessing for ICA (03-05) and improves performance for some algorithms.

In [ ]:
pca_white = PCA(n_components=20, whiten=True)
pca_white.fit(X_train_d)
Z_white = pca_white.transform(X_train_d)

pca_nw = PCA(n_components=20, whiten=False)
pca_nw.fit(X_train_d)
Z_nw = pca_nw.transform(X_train_d)

print("Whitening comparison (k=20, Digits training set):")
var_nw = Z_nw.var(axis=0)
var_wh = Z_white.var(axis=0)
print(f"  Non-whitened: var range [{var_nw.min():.3f}, {var_nw.max():.3f}]  "
      f"mean {var_nw.mean():.3f}")
print(f"  Whitened:     var range [{var_wh.min():.3f}, {var_wh.max():.3f}]  "
      f"mean {var_wh.mean():.3f}")
print("\nAfter whitening, all 20 components have unit variance (spherical distribution).")
print("See 03-05 (ICA) where whitening is a required pre-processing step.")

# Visualise the variance distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].bar(range(20), var_nw, color="#4472C4", edgecolor="black", linewidth=0.4)
axes[0].set_xlabel("PC index")
axes[0].set_ylabel("Variance")
axes[0].set_title("Non-whitened PC variances")

axes[1].bar(range(20), var_wh, color="#70AD47", edgecolor="black", linewidth=0.4)
axes[1].set_xlabel("PC index")
axes[1].set_ylabel("Variance")
axes[1].set_title("Whitened PC variances (all ≈ 1)")
plt.tight_layout()
plt.show()

### 4.7 MNIST Scree Plot & Compression Ratio

We compute the scree plot for MNIST (784 features) to show how much higher-dimensional
data still compresses well with PCA.

In [ ]:
X_mnist_c, mu_mnist = center_data(X_mnist)
cov_mnist            = compute_covariance_matrix(X_mnist_c)
eig_mnist, _         = eigendecompose_covariance(cov_mnist)
per_evr_mnist, cum_evr_mnist = compute_explained_variance_ratio(eig_mnist)

plot_scree(per_evr_mnist, cum_evr_mnist, "MNIST (28×28 pixels)", n_show=100)

k_90_mnist = int(np.searchsorted(cum_evr_mnist, 0.90)) + 1
k_95_mnist = int(np.searchsorted(cum_evr_mnist, 0.95)) + 1
print(f"MNIST: {k_90_mnist} components explain 90% of variance (out of 784 features)")
print(f"MNIST: {k_95_mnist} components explain 95% of variance (out of 784 features)")
print(f"Compression ratio at 90%: {784 / k_90_mnist:.1f}x")
print(f"Compression ratio at 95%: {784 / k_95_mnist:.1f}x")
print("\nMNIST is highly compressible because pixels are spatially correlated —")
print("neighboring pixels tend to have similar values.")

### 4.8 Per-Class Spread in PCA Space

We analyse how much variance each digit class occupies in the 2-D PCA projection.
High per-class variance (large spread) indicates a digit with high visual diversity
(e.g., digit "1" can be written many ways). Low spread indicates uniformity.

In [ ]:
def compute_per_class_pca_spread(
    Z: np.ndarray,
    y: np.ndarray,
    n_components: int = 2,
) -> pd.DataFrame:
    """Compute per-class spread statistics in a PCA-projected space.

    Measures the average distance from each point to its class centroid
    (intra-class spread) and the centroid's distance to the global origin.

    Args:
        Z: PCA-projected data of shape (n_samples, n_components).
        y: Class labels of shape (n_samples,).
        n_components: Number of PCA components used (for display).

    Returns:
        DataFrame with per-class spread statistics.
    """
    global_centroid = Z.mean(axis=0)
    classes         = np.unique(y)
    rows            = []

    for cls in classes:
        mask       = y == cls
        Z_cls      = Z[mask]
        centroid   = Z_cls.mean(axis=0)
        # Intra-class spread: mean distance to class centroid
        intra_dist = np.mean(np.linalg.norm(Z_cls - centroid, axis=1))
        # Centroid distance from origin
        origin_dist = np.linalg.norm(centroid)
        # Variance along each PC
        var_pc1    = Z_cls[:, 0].var() if n_components >= 1 else 0.0
        var_pc2    = Z_cls[:, 1].var() if n_components >= 2 else 0.0
        rows.append({
            "Class":          int(cls),
            "Count":          int(mask.sum()),
            "Intra spread":   round(intra_dist, 4),
            "Origin dist":    round(origin_dist, 4),
            "Var PC1":        round(var_pc1, 4),
            "Var PC2":        round(var_pc2, 4),
        })
    return pd.DataFrame(rows)


spread_df = compute_per_class_pca_spread(Z_2d_tr, y_train_d, n_components=2)
print("Per-class spread in 2-D PCA space (Digits training set):")
print(spread_df.to_string(index=False))

# Visualise intra-class spread as bar chart
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(spread_df["Class"], spread_df["Intra spread"],
       color=[colors[c] for c in spread_df["Class"]],
       edgecolor="black", linewidth=0.5)
ax.set_xlabel("Digit Class")
ax.set_ylabel("Intra-class Spread (mean dist to centroid)")
ax.set_title("Per-Class Spread in 2-D PCA Space — Digits")
ax.set_xticks(range(10))
plt.tight_layout()
plt.show()
print("Digit 1 tends to be compact (vertical stroke); 8 tends to spread more.")

### 4.9 PCA Denoising on MNIST

We repeat the denoising experiment on MNIST (784 features) to show that PCA
denoising generalises to higher dimensions and that more components are needed
to achieve similar reconstruction quality.

In [ ]:
# ── Fit full PCA on MNIST ─────────────────────────────────────────────────────
pca_mnist_full = PCA(n_components=100)
pca_mnist_full.fit(X_mnist)

X_mnist_noisy = add_gaussian_noise(X_mnist, noise_std=NOISE_STD, seed=SEED + 1)
mse_noisy_mnist = np.mean((X_mnist - X_mnist_noisy) ** 2)
print(f"MNIST denoising (noise_std={NOISE_STD}):")
print(f"  Noisy MSE: {mse_noisy_mnist:.4f}")

for k_dn in [10, 30, 50, 80, 100]:
    X_den_m = pca_denoise(X_mnist_noisy, n_components=k_dn, pca_clean=pca_mnist_full)
    mse_den_m = np.mean((X_mnist - X_den_m) ** 2)
    print(f"  k={k_dn:<3}  Denoised MSE: {mse_den_m:.4f}")

# Visualise 4 MNIST examples: clean / noisy / denoised at k=50
n_show_mnist = 4
k_best_mnist = 50
fig, axes = plt.subplots(3, n_show_mnist, figsize=(9, 4))

for col_idx in range(n_show_mnist):
    idx_m = col_idx
    axes[0, col_idx].imshow(
        X_mnist[idx_m].reshape(28, 28), cmap="gray", vmin=0, vmax=1
    )
    axes[0, col_idx].axis("off")
    if col_idx == 0:
        axes[0, col_idx].set_ylabel("Clean", fontsize=8)

    axes[1, col_idx].imshow(
        X_mnist_noisy[idx_m].reshape(28, 28), cmap="gray", vmin=0, vmax=1
    )
    axes[1, col_idx].axis("off")
    if col_idx == 0:
        axes[1, col_idx].set_ylabel(f"Noisy (σ={NOISE_STD})", fontsize=8)

    X_den_show = pca_denoise(
        X_mnist_noisy[idx_m:idx_m + 1], n_components=k_best_mnist,
        pca_clean=pca_mnist_full
    )
    axes[2, col_idx].imshow(
        np.clip(X_den_show[0].reshape(28, 28), 0, 1), cmap="gray", vmin=0, vmax=1
    )
    axes[2, col_idx].axis("off")
    if col_idx == 0:
        axes[2, col_idx].set_ylabel(f"Denoised k={k_best_mnist}", fontsize=8)

plt.suptitle("PCA Denoising on MNIST — Clean / Noisy / Denoised", fontsize=10)
plt.tight_layout()
plt.show()

### 4.10 kNN Decision Boundaries in 2-D PCA Space

To visualise classification difficulty in the 2-D projection, we plot kNN decision
boundaries. Overlapping regions indicate where the linear projection loses discriminative
information.

In [ ]:
def plot_knn_boundaries_2d(
    Z_train: np.ndarray,
    y_train: np.ndarray,
    Z_test: np.ndarray,
    y_test: np.ndarray,
    k_nn: int = 5,
    n_grid: int = 200,
) -> None:
    """Plot kNN decision boundaries in a 2-D PCA projection.

    Args:
        Z_train: 2-D PCA training scores, shape (n_train, 2).
        y_train: Training labels.
        Z_test: 2-D PCA test scores, shape (n_test, 2).
        y_test: Test labels.
        k_nn: Number of neighbours for kNN.
        n_grid: Grid resolution for boundary visualisation.
    """
    knn = KNeighborsClassifier(n_neighbors=k_nn)
    knn.fit(Z_train, y_train)
    test_acc = knn.score(Z_test, y_test)

    x_min, x_max = Z_train[:, 0].min() - 1, Z_train[:, 0].max() + 1
    y_min, y_max = Z_train[:, 1].min() - 1, Z_train[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, n_grid),
        np.linspace(y_min, y_max, n_grid),
    )
    grid_preds = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.contourf(xx, yy, grid_preds, alpha=0.25, cmap="tab10",
                levels=np.arange(-0.5, 10.5, 1))
    for cls in range(10):
        mask_tr = y_train == cls
        ax.scatter(Z_train[mask_tr, 0], Z_train[mask_tr, 1],
                   s=12, alpha=0.7, color=colors[cls], label=str(cls))
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"kNN (k={k_nn}) Decision Boundaries in 2-D PCA Space "
                 f"— Test Acc: {test_acc:.2%}")
    ax.legend(title="Digit", loc="upper right", fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()
    print(f"kNN test accuracy in 2-D PCA space: {test_acc:.2%}")
    print("Many classes overlap in 2-D — higher k is needed for good classification.")


plot_knn_boundaries_2d(Z_2d_tr, y_train_d, Z_2d_te, y_test_d, k_nn=5, n_grid=150)

### 4.11 Comprehensive Results Summary

We compile a reference table comparing PCA at different dimensionalities across all
key metrics: kNN accuracy, reconstruction MSE, and cumulative explained variance.

In [ ]:
def build_pca_summary_table(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    k_values: list,
    k_nn: int = 5,
) -> pd.DataFrame:
    """Build a comprehensive summary table for PCA at multiple dimensionalities.

    Computes kNN accuracy, reconstruction MSE, cumulative EVR, and compression
    ratio for each k, plus a row for the raw (unprojected) baseline.

    Args:
        X_train: Training data, shape (n_train, n_features).
        y_train: Training labels.
        X_test: Test data, shape (n_test, n_features).
        y_test: Test labels.
        k_values: List of component counts to evaluate.
        k_nn: Number of neighbours for kNN evaluation.

    Returns:
        DataFrame with one row per k and a baseline row.
    """
    n_features = X_train.shape[1]
    pca_max    = PCA(n_components=max(k_values))
    pca_max.fit(X_train)
    rows       = []

    # Baseline — raw features
    knn_raw  = KNeighborsClassifier(n_neighbors=k_nn)
    knn_raw.fit(X_train, y_train)
    acc_raw  = knn_raw.score(X_test, y_test)
    rows.append({
        "k":              "raw",
        "kNN Acc (%)":    round(acc_raw * 100, 2),
        "Recon MSE":      0.0,
        "Cum EVR (%)":    100.0,
        "Compression":    "1.0x",
    })

    for k in k_values:
        pca_k = PCA(n_components=k)
        pca_k.components_         = pca_max.components_[:k]
        pca_k.explained_variance_ = pca_max.explained_variance_[:k]
        pca_k.mean_               = pca_max.mean_
        pca_k.whiten              = False
        pca_k.is_fitted_          = True

        Z_tr = pca_k.transform(X_train)
        Z_te = pca_k.transform(X_test)
        X_r  = pca_k.inverse_transform(Z_te)

        knn_k = KNeighborsClassifier(n_neighbors=k_nn)
        knn_k.fit(Z_tr, y_train)
        acc_k = knn_k.score(Z_te, y_test)
        mse_k = np.mean((X_test - X_r) ** 2)
        evr_k = pca_max.cumulative_variance_ratio_[k - 1] * 100

        rows.append({
            "k":              int(k),
            "kNN Acc (%)":    round(acc_k * 100, 2),
            "Recon MSE":      round(mse_k, 3),
            "Cum EVR (%)":    round(evr_k, 1),
            "Compression":    f"{n_features / k:.1f}x",
        })
    return pd.DataFrame(rows)


summary_df = build_pca_summary_table(
    X_train_d, y_train_d, X_test_d, y_test_d,
    k_values=[2, 5, 10, 20, 30, 40, 64],
)
print("PCA Comprehensive Summary — Digits (kNN k=5):")
print()
print(summary_df.to_string(index=False))
print("\nk=20 offers the best trade-off: 87%+ accuracy at 3.2x compression.")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **PCA maximises variance via covariance eigendecomposition.** The $k$ eigenvectors
   with the largest eigenvalues form the optimal linear subspace; eigenvalue $\lambda_j$
   directly equals the variance projected onto the $j$-th principal component.

2. **Scree plots guide component selection.** Plotting cumulative EVR vs $k$ reveals
   diminishing returns; for Digits, just 20 components capture ~90% of variance,
   reducing 64 features by 3×. For MNIST (784 features), ~85 components achieve 90%.

3. **PCA denoises by discarding low-variance (noise) dimensions.** Fitting PCA on
   clean data and projecting noisy inputs removes noise concentrated in discarded
   components, with $k ≈ 20$ giving the best reconstruction MSE on Digits.

4. **Eigendecomposition PCA and truncated SVD are numerically equivalent.** Right
   singular vectors equal principal components, and $\lambda_j = \sigma_j^2/(n-1)$.
   SVD is preferred when $n < d$ since it avoids the $d \times d$ covariance matrix.

5. **PCA's core limitation is linearity.** It cannot separate classes on non-linear
   manifolds (make_moons). Non-linear methods (t-SNE, UMAP) address this at the cost
   of losing interpretability and the ability to map new points.

### What's Next

→ **03-04 (t-SNE, UMAP & Manifold Learning)** extends dimensionality reduction to
non-linear manifolds, achieving far better cluster separation on MNIST and FashionMNIST
at the cost of interpretability and global structure preservation.

### Going Further

- **Randomised SVD** (Halko et al., 2011): Approximates truncated SVD in $O(ndk)$ time —
  implemented in `sklearn.utils.extmath.randomized_svd`.
- **Kernel PCA** (03-08): Applies PCA in a kernel-induced feature space, capturing
  non-linear structure — directly addressing the limitation demonstrated in Section 4.5.
- **Autoencoders** (Module 11): Learn non-linear PCA-like compression via neural networks;
  the encoder and decoder generalise the `project` and `reconstruct` operations here.